In [ ]:
import math  # for mathematical operations
from copy import deepcopy  # for copying object

import numpy as np  # for numerical operations
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
import zipfile

%matplotlib inline

from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import train_test_split  # for splitting the dataset into training and testing sets
from sklearn.preprocessing import StandardScaler  # for scaling the data
from sklearn.preprocessing import MinMaxScaler  # for scaling the data
from sklearn.preprocessing import LabelEncoder  # for encoding the labels
from sklearn.preprocessing import OneHotEncoder  # for one-hot encoding the categorical variables

from sklearn.linear_model import LinearRegression  # for linear regression model
from sklearn.linear_model import LogisticRegression  # for logistic regression model
from sklearn.metrics import confusion_matrix  # for calculating confusion matrix for classification task

from sklearn.metrics import mean_squared_error  # for calculating mean squared error
from sklearn.metrics import mean_absolute_error  # for calculating mean absolute error
from sklearn.metrics import r2_score  # for calculating r2 score
from sklearn.metrics import accuracy_score  # for calculating accuracy score
from sklearn.metrics import precision_score  # for calculating precision score
from sklearn.metrics import recall_score  # for calculating recall score
from sklearn.metrics import f1_score  # for calculating f1 score

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import SGD



In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
# Task 1: Write your code here:
csv_path = os.path.join(path, "Q3_data.csv")
df = pd.read_csv(csv_path)

In [ ]:
# Task 2: Write your code here:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 1: Write your code here:
# Task 2: Write your code here:
# 2. Do we have missing values?
def check_missing_values(df):
    missing_values = df.isnull().sum()
    print("Missing Values per Column:")
    print(missing_values[missing_values > 0])
    if missing_values.any():
        print("\nHandle Missing Values as needed.")
    else:
        print("\nNo Missing Values Found.")

check_missing_values(df)

In [ ]:
stat_cols = ['P_2', 'B_2', 'D_144', 'D_41', 'D_142', 'D_145']

# TODO: Drop rows with missing values (not encluding 'type2')
df_clean =  df.dropna(subset=stat_cols,inplace=True)

In [ ]:
# Task 2: Write your code here:
# Task 3: Write your code here:
duplicates = df.duplicated().sum()
print(f"Number of Duplicate Samples: {duplicates}")
if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
else:
    print("No Duplicate Samples Found.")

In [ ]:
# Task 3: Write your code here:
# Task 4: Write your code here:

from sklearn.preprocessing import LabelEncoder

categorical_cols = df.select_dtypes(include=["object"]).columns
for col in categorical_cols:
  print(f"Encoding column: {col}")
  le = LabelEncoder()
  df[col] = le.fit_transform(df[col])

df

In [ ]:
# Task 4: Write your code here:
# Task 5: Write your code here:
# Standardize features using StandardScaler
numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.drop("Target")  ### DON'T SCALE THE TARGET
scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])
df

In [ ]:
# Task 5: Write your code here:
#ee
#Is the target imbalanced?
import seaborn as sns
def check_target_imbalance(df, target_column):

  print("Target Distribution:")
  print(df[target_column].value_counts(normalize=True))
  sns.countplot(x=df[target_column])
  plt.title("Target Distribution")
  plt.show()

check_target_imbalance(df,"Target")

In [ ]:
# Task 1: Write your code here:
# Task 1: Write your code here:
X = df.drop("Target", axis=1).astype(float)
y = df['Target'].astype(float)

In [ ]:
# Task 2,3,4,5: Write your code here:

from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=5,shuffle=True)

for train_index, test_index in skf.split(X, y):

    # Split data into training and testing sets
    X_Train, X_Test = X[train_index], X[test_index]
    y_Train, y_Test = y[train_index], y[test_index]

In [ ]:
# Define classification models
from catboost import CatBoostClassifier
models = {
    "CatBoost Classifier": CatBoostClassifier(verbose=0)
}

models["Ensemble"] = VotingClassifier(estimators=[
        ('lr', models["CatBoost Classifier"]), ('rf', models["Logistic Regression"]), ('gnb', models["Random Forest Classifier"])], voting="soft")

for model_name, model in models.items():
    scores_accuracy = []
    scores_precision = []
    scores_recall = []
    scores_f1 = []

    # Stratified 5-Fold Cross-Validation
    skf = KFold(n_splits=5)
    for train_index, test_index in skf.split(X, y):
        # Split data into training and testing sets
        X_Train, X_Test = X.loc[train_index, :], X.loc[test_index, :]
        y_Train, y_Test = y.iloc[train_index], y.iloc[test_index]
        # Train the model
        model.fit(X_Train, y_Train)
        # Predict on the test set
        y_pred = model.predict(X_Test)

        # Calculate metrics
        scores_f1.append(f1_score(y_Test, y_pred, average='weighted'))

    # Print the results
    print(f"{model_name} F1-Score: {np.mean(scores_f1):.4f}")
    print("\n")


In [ ]:
# Plotting results
plt.figure(figsize=(7, 5))

plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Validation Loss')
plt.title('Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Task 1: Write your code here:
# Retrieve CatBoost feature importances and sort them
catboost_model = model["CatBoost Classifier"]
catboost_importance = list(zip(X.columns, catboost_model.feature_importances_))
sorted_catboost_importance = sorted(catboost_importance, key=lambda x: x[1], reverse=True)

# Extract features and their importances
features, importances = zip(*sorted_catboost_importance)

# Plot feature importances
plt.figure(figsize=(18, 14))
plt.barh(features, importances, color='orange')
plt.xlabel('Importance Score')
plt.ylabel('Features')
plt.title('CatBoost Feature Importance')
plt.gca().invert_yaxis()  # Invert y-axis to show the most important features at the top
plt.show()


In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here: